# Generate mock data with `itk-mock`

This notebook uses **`packages/itk-mock`** (`import itk_mock`), not the legacy
`intelligence_toolkit.generate_mock_data` module or the Streamlit workflow. It reads one
existing repository input directly, without copying it or writing new files:

- `example_outputs/generate_mock_data/customer_complaints/customer_complaints_schema.json`

The workflow has three parts: a **JSON schema** describes the records, a model generates
**records** that follow it (one unseeded batch, then batches seeded from existing records
so that near duplicates and close relations can be requested), and a model writes one
**text** per record.

The model sits behind a small async protocol, `itk_mock.ChatFn`. This notebook defines an
**offline, deterministic chat function** (step 5) and uses it everywhere, so every cell
runs without an API key or network access and gives the same output on every run. Step 9
is a clearly marked cell showing how to swap in the real OpenAI client.

## 1. Install and select a Python kernel

Use **Python 3.11 or 3.12** (the packages require `>=3.11,<3.13`). From the repository
root, run once in a terminal:

```powershell
uv venv --python 3.12 .venv-mock-demo
uv pip install --python .venv-mock-demo\Scripts\python.exe -e .\packages\itk-core -e .\packages\itk-mock ipykernel
.\.venv-mock-demo\Scripts\python.exe -m ipykernel install --user --name itk-mock-demo --display-name "Python (itk-mock demo)"
```

On macOS/Linux use `.venv-mock-demo/bin/python` and slash-separated paths. Install **both
local packages together**: `itk-core` on PyPI is a different project, and these packages
are bundled in the Intelligence Toolkit wheel rather than published on their own. If you
already use the repository's own environment (`uv sync --extra dev`), its kernel works
too.

Open this notebook in VS Code or Jupyter, select the kernel, and run all cells. Jupyter's
top-level `await` runs the async API directly. Input lookup walks up from the working
directory, so the notebook runs from the repository root or from its own folder.

In [ ]:
# Copyright (c) 2026 Microsoft Corporation. All rights reserved.
# Licensed under the MIT license. See LICENSE file in the project.

import ast
import hashlib
import json
import random
import re
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import polars as pl
from IPython.display import Markdown
from IPython.display import display as show
from itk_mock import (
    ArrayFieldType,
    DuplicateFieldError,
    MockConfig,
    MockDataGenerator,
    PrimitiveFieldType,
    SchemaPathError,
    StringFormat,
    TextConfig,
)
from itk_mock import schema as sb
from itk_mock.schema import extract_array_fields
from itk_mock.template import (
    evaluate_object_and_schema,
    evaluate_schema,
    generate_object_from_schema,
)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(14)
pl.Config.set_fmt_str_lengths(80)

cwd = Path.cwd().resolve()
relative_input = (
    Path("example_outputs")
    / "generate_mock_data"
    / "customer_complaints"
    / "customer_complaints_schema.json"
)
repo_root = next(
    (parent for parent in (cwd, *cwd.parents) if (parent / relative_input).is_file()),
    None,
)
if repo_root is None:
    message = (
        "Run from within the repository containing the customer-complaints schema."
    )
    raise FileNotFoundError(message)
input_path = repo_root / relative_input
print("Input:", input_path.relative_to(repo_root))


def installed_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:  # dev-mode checkout without distribution metadata
        return "dev"


packages = ["itk-mock", "itk-core", "polars", "pydantic", "jsonschema"]
show(
    pl.DataFrame({
        "package": packages,
        "version": [installed_version(name) for name in packages],
    })
)

## 2. Load and validate the existing schema

A schema is a plain JSON Schema `dict`. `evaluate_schema` builds the emptiest object the
schema describes and validates the schema against it: `VALID`, `SCHEMA_INVALID` (not a
JSON Schema) or `OBJECT_INVALID` (the schema is fine but its own template breaks a
constraint such as `minLength`). `extract_array_fields` lists the label path of every
array; the **first** one is the primary record array, the one record counts refer to.

In [ ]:
complaints_schema = json.loads(input_path.read_text(encoding="utf-8"))

print("Title:", complaints_schema["title"])
print("Validation:", evaluate_schema(complaints_schema).name)
record_arrays = extract_array_fields(complaints_schema)
print("Record arrays:", record_arrays)

fields = sb.get_properties(complaints_schema, record_arrays[0])
show(Markdown(f"**Fields of `{'.'.join(record_arrays[0])}`**"))
show(
    pl.DataFrame({
        "field": list(fields),
        "type": [spec["type"] for spec in fields.values()],
        "enum": [", ".join(spec.get("enum", [])) for spec in fields.values()],
        "required": [
            name in sb.get_required_list(complaints_schema, record_arrays[0])
            for name in fields
        ],
        "description": [spec.get("description", "") for spec in fields.values()],
    })
)

## 3. Build a small schema from scratch

The builder functions in `itk_mock.schema` edit a schema in place. Each takes the schema,
a **nesting** path of field labels leading to the object (or array of objects) being
edited, and usually a field label. `create_boilerplate_schema` starts with one empty
`records` array, so its fields live at `["records"]`.

Below: a support-ticket schema with strings, a constrained number, an enum, a boolean, a
nested object, an array of strings and a nested array of objects. New fields are required
by default, which is what OpenAI structured outputs need.

In [ ]:
tickets_schema = sb.create_boilerplate_schema(
    title_field="Support tickets",
    description_field="A small schema built with the itk-mock builder functions",
)
RECORDS = ["records"]

sb.add_primitive_field(tickets_schema, RECORDS, "ticket_id", "Identifier of the ticket")
sb.add_primitive_field(
    tickets_schema, RECORDS, "email", "Contact email of the customer"
)
sb.set_string_format(tickets_schema, RECORDS, "email", StringFormat.EMAIL)
sb.add_primitive_field(tickets_schema, RECORDS, "priority", "How urgent the ticket is")
sb.set_enum_values(tickets_schema, RECORDS, "priority", ["low", "medium", "high"])
sb.add_primitive_field(
    tickets_schema,
    RECORDS,
    "days_open",
    "Days since the ticket was opened",
    field_type=PrimitiveFieldType.NUMBER,
)
sb.set_number_minimum(tickets_schema, RECORDS, "days_open", 0)
sb.set_number_maximum(tickets_schema, RECORDS, "days_open", 30)
sb.add_primitive_field(
    tickets_schema,
    RECORDS,
    "resolved",
    "Whether the ticket is closed",
    field_type=PrimitiveFieldType.BOOLEAN,
)

sb.add_object_field(tickets_schema, RECORDS, "customer", "Who raised the ticket")
sb.add_primitive_field(tickets_schema, ["records", "customer"], "name", "Customer name")
sb.add_primitive_field(tickets_schema, ["records", "customer"], "city", "Customer city")

sb.add_array_field(tickets_schema, RECORDS, "tags", "Free-form labels")
sb.add_array_field(
    tickets_schema,
    RECORDS,
    "messages",
    "Messages exchanged on the ticket",
    item_type=ArrayFieldType.OBJECT,
)
sb.add_primitive_field(
    tickets_schema, ["records", "messages"], "author", "Who wrote it"
)
sb.add_primitive_field(tickets_schema, ["records", "messages"], "body", "Message text")

# Editing: reorder, rename, and a field added without a label gets a numbered default.
sb.move_field_up(tickets_schema, RECORDS, "priority")
sb.rename_field(tickets_schema, RECORDS, "days_open", "open_days")
placeholder = sb.add_primitive_field(tickets_schema, RECORDS)
print("Auto-labelled field:", placeholder)
sb.delete_field(tickets_schema, RECORDS, placeholder)

print("Fields:", list(sb.get_properties(tickets_schema, RECORDS)))
print("Required:", sb.get_required_list(tickets_schema, RECORDS))
print("Record arrays:", extract_array_fields(tickets_schema))
print("Validation:", evaluate_schema(tickets_schema).name)

Edits that cannot be applied raise a typed error instead of doing nothing: a duplicate
label is a `DuplicateFieldError`, a nesting path that does not lead to an object is a
`SchemaPathError`. Both are `SchemaEditError`s, and the schema is left unchanged.

In [ ]:
before = json.dumps(tickets_schema)
for attempt in (
    lambda: sb.add_primitive_field(tickets_schema, RECORDS, "email"),
    lambda: sb.add_primitive_field(tickets_schema, ["records", "email"], "domain"),
    lambda: sb.rename_field(tickets_schema, RECORDS, "ticket_id", "priority"),
):
    try:
        attempt()
    except (DuplicateFieldError, SchemaPathError) as error:
        print(f"{type(error).__name__}: {error}")
assert json.dumps(tickets_schema) == before

## 4. The template object

`generate_object_from_schema` returns the emptiest object a schema allows: empty strings,
`False`, the lowest allowed number, the first enum value, empty arrays of primitives and
one template element in each array of objects. It is what `evaluate_schema` validates,
and a quick way to see the shape of what the model will be asked for.

In [ ]:
template = generate_object_from_schema(tickets_schema)
print(json.dumps(template, indent=2))
print(
    "Template valid for schema:",
    evaluate_object_and_schema(template, tickets_schema).name,
)

## 5. An offline, deterministic chat function

`itk_mock.ChatFn` is any async callable taking the chat `messages`, an optional
`response_schema` (the JSON schema the reply must follow; `None` means free text) and an
optional `temperature`, and returning the reply text.

`offline_chat` below stands in for the model. It reads the same prompts the real model
would get and answers from them alone:

- **Record prompts** (`response_schema` given): it walks the schema and fills every field
  from small word lists, producing as many primary records as the prompt asks for. When
  the prompt carries a seed record, the first records are *near duplicates* of the seed
  (one text field upper-cased or cut short) and the next are *close relations* (new records sharing two of
  the seed's text fields), in the numbers the prompt requests.
- **Text prompts** (no schema): it writes one sentence per field of the input record.

All randomness comes from a `random.Random` seeded with a hash of the prompt, so the same
prompt always gets the same reply. Nothing here is specific to one schema.

In [ ]:
WORDS = {
    "name": [
        "Ana Silva",
        "Bo Chen",
        "Cara Diaz",
        "Dev Patel",
        "Eli Novak",
        "Fay Okoro",
        "Gus Lind",
        "Hana Sato",
        "Ivo Marek",
        "Jia Wong",
    ],
    "city": ["Lakeside", "Hilltop", "Riverbend", "Marshfield"],
    "street": ["12 Mill Lane", "480 Harbor Road", "7 Orchard Row", "93 Quarry Street"],
    "quarter": ["2023-Q1", "2023-Q2", "2023-Q3", "2023-Q4"],
    "author": ["customer", "agent"],
    "body": [
        "The order arrived late.",
        "We are looking into it.",
        "Thanks for the update.",
    ],
    "tags": ["billing", "delivery", "refund", "login"],
}


def section(prompt: str, title: str) -> str:
    # Text between "--<title>--" and the next "--...--" header.
    return prompt.split(f"--{title}--", 1)[1].split("\n--", 1)[0].strip()


def fake_value(name: str, spec: dict, rng: random.Random):
    kind = spec.get("type")
    if spec.get("enum"):
        return rng.choice(spec["enum"])
    if kind == "object":
        return fake_record(spec, rng)
    if kind == "array":
        return [fake_value(name, spec["items"], rng) for _ in range(rng.randint(1, 2))]
    if kind == "boolean":
        return rng.random() < 0.3
    if kind == "number":
        return rng.randint(int(spec.get("minimum", 18)), int(spec.get("maximum", 80)))
    if name in WORDS:
        return rng.choice(WORDS[name])
    return f"{name[:3].upper()}-{rng.randint(1000, 9999)}"


def fake_record(spec: dict, rng: random.Random) -> dict:
    record = {
        name: fake_value(name, sub, rng) for name, sub in spec["properties"].items()
    }
    owner = record.get("name") or (record.get("customer") or {}).get("name")
    if "email" in record and owner:
        record["email"] = owner.lower().replace(" ", ".") + "@example.com"
    return record


def text_fields(spec: dict) -> list[str]:
    return [
        name
        for name, sub in spec["properties"].items()
        if sub.get("type") == "string" and not sub.get("enum")
    ]


def near_duplicate(seed: dict, spec: dict, rng: random.Random) -> dict:
    record = dict(seed)
    name = rng.choice(text_fields(spec))
    value = str(record[name])
    record[name] = value.upper() if rng.random() < 0.5 else value[:-1]
    return record


def close_relation(seed: dict, spec: dict, rng: random.Random) -> dict:
    record = fake_record(spec, rng)
    for name in rng.sample(text_fields(spec), 2):
        record[name] = seed[name]
    return record


def describe(record: dict) -> str:
    parts = [f"{key.replace('_', ' ')} is {value}" for key, value in record.items()]
    return "In this record, " + "; ".join(parts) + "."


# The signature is ChatFn's: async, and a temperature this fake has no use for.
async def offline_chat(messages, response_schema=None, temperature=None) -> str:  # noqa: ARG001, RUF029
    prompt = messages[0]["content"]
    rng = random.Random(int(hashlib.sha256(prompt.encode()).hexdigest()[:16], 16))
    if response_schema is None:
        return describe(json.loads(section(prompt, "Input text")))

    primary = ast.literal_eval(section(prompt, "Primary Record Array"))
    targets = section(prompt, "Record Targets")
    total = int(re.search(r"(?:Output record count|Total records): (\d+)", targets)[1])
    record_spec = sb.get_container(response_schema, primary)

    records = []
    if "--Seed Record--" in prompt:
        seed = ast.literal_eval(section(prompt, "Seed Record"))
        duplicates = int(re.search(r"Near duplicates of seed: (\d+)", targets)[1])
        relations = int(re.search(r"Close relations of seed: (\d+)", targets)[1])
        records += [near_duplicate(seed, record_spec, rng) for _ in range(duplicates)]
        records += [close_relation(seed, record_spec, rng) for _ in range(relations)]
    records += [fake_record(record_spec, rng) for _ in range(total - len(records))]

    reply = fake_record(response_schema, rng)
    holder = reply
    for label in primary[:-1]:
        holder = holder[label]
    holder[primary[-1]] = records
    return json.dumps(reply)


preview = await offline_chat(
    [
        {
            "role": "system",
            "content": "--Primary Record Array--\n\n['records']\n\n--Record Targets--\n\nOutput record count: 1\n",
        }
    ],
    response_schema=tickets_schema,
)
print(json.dumps(json.loads(preview), indent=2))

## 6. Generate records

`MockDataGenerator` holds the chat function. `MockConfig` carries the knobs of the
workflow's *Generate mock records* tab plus an explicit `seed`:

| Field | Meaning |
| ----- | ------- |
| `num_records_overall` | Total primary records to ask for |
| `records_per_batch` | Primary records per model call |
| `duplicate_records_per_batch` | Near duplicates of the seed record in each seeded batch |
| `related_records_per_batch` | Close relations of the seed record in each seeded batch |
| `parallel_batches` | Seeded batches generated concurrently per round |
| `temperature`, `generation_guidance` | Passed to the model |
| `seed` | Seeds the choice of seed records, so a run can be repeated |

The first batch is unseeded. Each later round samples existing primary records, one per
batch, and asks for a batch built around each. Here: 40 complaints in batches of 10, so
one unseeded call and three seeded calls, each asked for two near duplicates and one
close relation of its seed. `on_update` is called with the frames after every round.

In [ ]:
generator = MockDataGenerator(offline_chat)
config = MockConfig(
    num_records_overall=40,
    records_per_batch=10,
    duplicate_records_per_batch=2,
    related_records_per_batch=1,
    temperature=0.5,
    generation_guidance="Complaints from customers of a regional appliance retailer",
    seed=42,
)

progress = []
result = await generator.generate_records(
    complaints_schema,
    config,
    on_update=lambda frames: progress.append(frames["customer_complaints"].height),
)

print("Records after each round:", progress)
print("Model replies kept:", len(result.generated_objects))
print("Frames:", {name: frame.shape for name, frame in result.frames.items()})
print(
    "Merged object valid for schema:",
    evaluate_object_and_schema(result.json_object, complaints_schema).name,
)

complaints = result.primary_frame
assert complaints.height == config.num_records_overall
show(Markdown(f"**First 12 of {complaints.height} generated records**"))
show(complaints.head(12))

The result keeps everything later steps need, so nothing has to be generated twice:

- `result.json_object`: the merged JSON object (what the workflow offers for download)
- `result.frames`: one polars frame per record array, keyed by its dotted path
- `result.generated_objects`: every parsed model reply, in generation order
- `result.conflicts`: paths of top-level scalars that differed between batches

Records 10 to 12 below are the start of the first seeded batch: two near duplicates of
the seed record (the same person with one text field altered: upper-cased or cut short) and one close relation (a new
person sharing two of the seed's fields).

In [ ]:
first_batch = result.generated_objects[0]["customer_complaints"]
seeds = random.Random(config.seed).sample(first_batch, 3)  # the same draw the run made
seeded_batch = result.generated_objects[1]["customer_complaints"]

show(Markdown("**Seed record of the first seeded batch**"))
show(pl.from_dicts(seeds[:1]))
show(Markdown("**Its two near duplicates and one close relation**"))
show(pl.from_dicts(seeded_batch[:3]))

### Reproducibility and nested arrays

With the same `seed` (or the same `random.Random` passed as `rng`) the same seed records
are chosen, so a deterministic model gives an identical object. A different seed picks
different seed records.

The support-ticket schema from step 3 has nested objects and arrays. Nested objects
become dotted columns (`customer.name`); every array gets its own frame
(`records.messages` holds every message of every ticket); arrays also stay on their
parent as list columns.

In [ ]:
again = await generator.generate_records(complaints_schema, config)
other = await generator.generate_records(
    complaints_schema, config.model_copy(update={"seed": 7})
)
print("Same seed, same object:", again.json_object == result.json_object)
print("Different seed, same object:", other.json_object == result.json_object)

tickets = await generator.generate_records(
    tickets_schema, MockConfig(num_records_overall=6, records_per_batch=3, seed=1)
)
for name, frame in tickets.frames.items():
    show(Markdown(f"**`{name}`** ({frame.height} rows)"))
    show(frame.head(6))

## 7. Generate texts from the records

`generate_texts` takes a polars frame (or a list of record dicts) and asks for one text
per row, `parallelism` at a time, in row order. It runs on the frames already held by
`result`, or on any CSV loaded later, without touching record generation. The result
keeps the input JSON shown to the model next to each text.

In [ ]:
calls_before = len(result.generated_objects)
texts = await generator.generate_texts(
    complaints.head(8),
    TextConfig(
        generation_guidance="Write as the customer, in the first person",
        temperature=0.5,
        parallelism=4,
    ),
)
assert len(result.generated_objects) == calls_before  # records were not regenerated

show(Markdown("**Generated texts (`texts.frame`)**"))
show(texts.frame)
print("Input for the first text:")
print(texts.input_texts[0])

## 8. Export

Nothing is written by this notebook. To save what the workflow offers for download:

```python
Path("customer_complaints.json").write_text(json.dumps(result.json_object, indent=2))
result.primary_frame.write_csv("customer_complaints_records.csv")
texts.frame.write_csv("customer_complaints_texts.csv")
```

Frames with list columns (arrays kept on their parent) need those columns dropped or
converted to text before `write_csv`.

## 9. Swap in the real OpenAI client

> **This is the only cell that calls a model.** It is switched off by default so the
> notebook runs offline. Set `USE_OPENAI = True` to run it.

`MockDataGenerator.from_openai` wraps itk-core's `OpenAIClient`. With no argument,
`OpenAIConfiguration` reads `OPENAI_API_KEY`, `OPENAI_API_MODEL` and, for Azure,
`OPENAI_TYPE`, `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_VERSION` from the environment;
pass a dict to set them explicitly. The model must support structured outputs. Imported
schemas should go through `normalize_schema_for_openai` first, which marks every property
required and closes every object, as strict structured outputs demand.

Everything else is unchanged: same schema, same config, same result types.

In [ ]:
USE_OPENAI = False  # set to True to call the real model (needs OPENAI_API_KEY)

if USE_OPENAI:
    from itk_core.openai_configuration import OpenAIConfiguration

    openai_generator = MockDataGenerator.from_openai(OpenAIConfiguration())
    openai_schema = sb.normalize_schema_for_openai(
        json.loads(json.dumps(complaints_schema))
    )
    openai_result = await openai_generator.generate_records(
        openai_schema,
        MockConfig(
            num_records_overall=20,
            records_per_batch=10,
            duplicate_records_per_batch=1,
            related_records_per_batch=1,
            seed=42,
        ),
    )
    show(openai_result.primary_frame)
    openai_texts = await openai_generator.generate_texts(
        openai_result.primary_frame.head(3), TextConfig(parallelism=3)
    )
    show(openai_texts.frame)
else:
    print("Skipped: USE_OPENAI is False, so no model was called.")